# shopagent walkthrough

`shop/` 코드를 step별로 뜯어보는 노트북입니다. 앞 단계는 LLM/GPU 없이 Stub으로 돌리고, 마지막에만 실제 `llama-server`를 붙입니다.

| Step | 내용 | LLM |
|---|---|---|
| 0 | import / 경로 | 없음 |
| 1 | mock DB 살펴보기 | 없음 |
| 2 | `Shop` 도구 직접 호출 | 없음 |
| 3 | 시나리오 + 프롬프트 | 없음 |
| 4 | Stub으로 한 턴 `run_turn` | Stub |
| 5 | grader 채점 | Stub |
| 6 | (선택) 실제 서버로 smoke 1개 | GPU |

커널은 `bench/shopagent/.venv` 를 쓰는 것을 권장합니다.

```sh
cd ~/llama.cpp/bench/shopagent
PYTHONDONTWRITEBYTECODE=1 .venv/bin/python -m ipykernel install --user --name shopagent --display-name shopagent
```

## Step -1. 터미널에서 서버 먼저 띄우기 (g12-native와 같은 설정)

WSL 터미널에서 아래 명령으로 Gemma 12B 서버를 띄워 두고, 이 노트북은 그 서버에 붙어서 씁니다. `shop/servers.py`의 `g12-native` 설정에서 그대로 뽑은 명령이라 측정할 때와 조건이 같습니다.

```sh
/home/home/llama.cpp/build/bin/llama-server \
  -m /home/home/llama.cpp/models/gemma-4-12b-it-Q4_K_M.gguf \
  --jinja -ngl 99 -ctk q8_0 -ctv q8_0 -fa on -np 1 -c 16384 \
  --host 127.0.0.1 --port 18380 --n-cpu-moe 0 \
  --chat-template-kwargs '{"enable_thinking": false}' --reasoning-budget 0
```

- 떴는지 확인: `curl http://127.0.0.1:18380/health`
- 다 쓰면 터미널에서 `Ctrl+C`로 꼭 끄기 (GPU를 잡고 있으면 다음 측정과 충돌)
- 띄워 둔 상태에서는 Step 6의 `RUN_REAL=True` 셀을 쓰지 마세요 (같은 포트로 서버를 하나 더 띄우려다 충돌)

노트북에서 붙는 법 (Step 0 import 이후):

```python
from shop.agents import LocalLLM
llm = LocalLLM(18380)
agents = Agents(llm, "C01")
result = run_turn(agents, Shop(BASE, "C01"), 0, "O004 반품 가능한지 알려줘", [])
print(result["routes"], result["reply"])
```


## Step 0. import

노트북이 `bench/shopagent/` 에 있으니 cwd만 맞으면 `from shop...` 가 됩니다.

In [8]:
from pathlib import Path
import json
import copy
import os

# 노트북 cwd가 shopagent가 아니면 여기로 맞춤
ROOT = Path("/home/home/llama.cpp/bench/shopagent").resolve()
os.chdir(ROOT)
print("cwd:", Path.cwd())

from langchain_core.messages import AIMessage, HumanMessage

from shop.shared import REFERENCE_DATE, AGENTS, load
from shop.tools import Shop, SCOPES, IRREVERSIBLE, schemas
from shop.agents import Agents
from shop.graph import run_turn
from shop.grader import grade_turn, grade_scenario, final_state

BASE = load(ROOT / "data/shop_db.json")
DATA = load(ROOT / "data/scenarios.json")
PROMPTS = load(ROOT / "data/prompts.json")
SCENARIOS = {s["id"]: s for s in DATA["scenarios"]}

print("reference_date:", REFERENCE_DATE)
print("agents:", AGENTS)
print("scenarios:", len(SCENARIOS), "smoke:", DATA["smoke_ids"])

cwd: /home/home/llama.cpp/bench/shopagent
reference_date: 2026-10-22
agents: ('product', 'order_delivery', 'refund_exchange', 'payment', 'qna', 'human')
scenarios: 60 smoke: ['single-01', 'single-07', 'single-18']


## Step 1. mock DB 살펴보기

`data/shop_db.json` 은 고객 / 상품 / 주문 / 결제 / 정책의 합성 데이터입니다. 시스템 시계 대신 기준일 `2026-10-22`만 씁니다.

In [ ]:
print("top keys:", list(BASE.keys()))
print("customers:", list(BASE["customers"]))
print("products:", len(BASE["products"]), "orders:", len(BASE["orders"]), "policies:", list(BASE["policies"]))

# 로그인 고객 C01의 주문 몇 개
for oid, order in BASE["orders"].items():
    if order["customer_id"] == "C01":
        print(oid, order["status"], "amount=", order["amount"], "delivered_at=", order.get("delivered_at"))

In [ ]:
# 반품 실험에 자주 쓰는 O004 / 상품 S01
print("O004:", json.dumps(BASE["orders"]["O004"], ensure_ascii=False, indent=2))
print("S01:", json.dumps(BASE["products"]["S01"], ensure_ascii=False, indent=2)[:800])
print("P01:", json.dumps(BASE["policies"]["P01"], ensure_ascii=False, indent=2))

## Step 2. `Shop` 도구 직접 호출

`Shop` 은 mock DB 위에서 결정적으로 동작합니다.
- 시나리오마다 `db_patch` / `faults` 로 상태를 덮어쓸 수 있음
- `invoke` 는 호출마다 before/after 변경을 `trace` 에 남김
- Mock 백엔드 API처럼 동작: 성공은 `{"ok": true, ...}`, 실패는 `{"ok": false, "error": 코드, "message": ...}`
- 주문번호를 받는 도구는 모두 소유자를 검사함 (타인 주문은 `FORBIDDEN`, 없는 주문은 `NOT_FOUND`)

In [ ]:
shop = Shop(BASE, customer_id="C01")

print("내 주문:", [o["order_id"] for o in shop.list_orders()])
print("search:", [(p["product_id"], p["name"], p["price"]) for p in shop.search_products("코튼 티셔츠")])
print("track O003:", shop.track_shipment("O003"))
print("eligible O004:", shop.check_refund_eligibility("O004"))
print("eligible O004 상품불량:", shop.check_refund_eligibility("O004", "상품불량"))
print("eligible O006 (기한 초과):", shop.check_refund_eligibility("O006"))

# 타인 주문은 서버가 거절 (FORBIDDEN)
print("타인 O013:", shop.get_order("O013"))
print("O008 부분취소:", shop.get_order("O008")["status"], [(i["item_id"], i["amount"], i["status"]) for i in shop.get_order("O008")["items"]])

In [ ]:
# 되돌릴 수 없는 행동 + DB 변경 추적
shop = Shop(BASE, "C01")
print("before:", shop.db["orders"]["O004"]["status"])

out = shop.invoke("request_refund", {"order_id": "O004", "reason": "단순변심"}, turn=0)
print("output:", out)
print("after:", shop.db["orders"]["O004"]["status"], shop.db["payments"]["O004"]["pending_refund"])
print("changes:", shop.trace[-1]["changes"])
print("irreversible tools:", sorted(IRREVERSIBLE))

# 원본 BASE는 안 바뀜
print("BASE O004 still:", BASE["orders"]["O004"]["status"])

In [ ]:
# fault 주입: 첫 호출만 오류, 두 번째는 정상
shop = Shop(
    BASE,
    "C01",
    db_patch={"orders": {"O003": {"status": "배송중"}}},
    faults={"track_shipment": {"call": 1, "output": {"ok": False, "error": "INTERNAL_ERROR", "message": "carrier timeout"}}},
)
print("1st:", shop.invoke("track_shipment", {"order_id": "O003"}, 0))
print("2nd:", shop.invoke("track_shipment", {"order_id": "O003"}, 0))
print("fault flags:", [t.get("fault_injected") for t in shop.trace])

In [ ]:
# 에이전트별 tool schema (LLM에 넘기는 OpenAI tools 형식)
for agent in AGENTS:
    if agent == "human":
        continue
    names = [s["function"]["name"] for s in schemas(agent)]
    print(f"{agent:16} {names}")
print("scopes:", {k: v for k, v in SCOPES.items()})

## Step 3. 시나리오 + 프롬프트

시나리오는 고정 사용자 대본 + gold expect + final_state 입니다. 런타임에는 gold를 모델에 보내지 않습니다.

In [ ]:
SID = "single-18"  # 바꿔가며 실험: single-01, single-07, chain-01, ...
scenario = SCENARIOS[SID]

print("id:", scenario["id"], "category:", scenario["category"], "customer:", scenario["customer_id"])
print("rationale:", scenario["gold_rationale"])
print("final_state.equals:", scenario["final_state"]["equals"])
print()
for i, turn in enumerate(scenario["turns"]):
    print(f"--- turn {i} ---")
    print("user:", turn["user"])
    print("route:", turn["expect"]["route"])
    print("required_calls:", turn["expect"]["required_calls"])
    print("ask_confirmation:", turn["expect"]["ask_confirmation"])
    print("must_include:", turn["expect"]["reply_must_include"])

In [ ]:
# 시스템 프롬프트 (common + 역할)
print("=== common (일부) ===")
print(PROMPTS["common"].format(customer_id="C01", reference_date=REFERENCE_DATE)[:500], "...")
print("\n=== supervisor ===")
print(PROMPTS["supervisor"])
print("\n=== refund_exchange ===")
print(PROMPTS["refund_exchange"][:600], "...")

## Step 4. Stub으로 한 턴 `run_turn`

그래프 흐름:

```
START -> supervisor -> agent_llm <-> tools -> END
                    \-> escalate -> END
```

`Stub` 은 미리 정해둔 AIMessage를 순서대로 돌려줍니다. LLM 없이 supervisor 라우팅 / 도구 호출 / 최종 답변을 재현할 수 있습니다.

In [ ]:
def message(text="", *calls):
    """tests/test_shop.py 와 같은 helper"""
    return AIMessage(
        content=text,
        tool_calls=[{"name": name, "args": args, "id": f"c-{i}", "type": "tool_call"} for i, (name, args) in enumerate(calls)],
    )


def routed(agent):
    return message("", ("transfer_to_" + agent, {}))


class Stub:
    def __init__(self, responses):
        self.responses = iter(responses)
        self.nodes = []

    def call(self, messages, tools, node):
        self.nodes.append(node)
        return next(self.responses), {
            "node": node,
            "ttft_ms": None,
            "content_ttft_ms": None,
            "total_ms": 0,
            "tokens": None,
        }


print("helpers ready")

In [ ]:
# 예 A: 상품 조회 한 턴 (single-01 느낌)
stub = Stub([
    routed("product"),
    message("", ("search_products", {"query": "코튼 티셔츠"})),
    message("코튼 티셔츠는 30000원입니다."),
])
shop = Shop(BASE, "C01")
result = run_turn(Agents(stub, "C01"), shop, turn=0, user="코튼 티셔츠 찾아줘. 이름과 가격 알려줘.", history=[])

print("routes:", result["routes"])
print("agent:", result["agent"])
print("steps:", result["steps"], "tool_count:", result["tool_count"])
print("nodes called:", stub.nodes)
print("reply:", result["reply"])
print("tool trace:")
for t in shop.trace:
    print(" -", t["name"], t["arguments"], "->", t["output"] if isinstance(t["output"], dict) and "error" in t["output"] else "(ok)")

In [ ]:
# 예 B: 반품 가능 확인 후 동의 질문 (single-18 turn 0)
stub = Stub([
    routed("refund_exchange"),
    message("", ("get_order", {"order_id": "O004"}), ("check_refund_eligibility", {"order_id": "O004"})),
    # 위 셀은 tool_calls를 한 AIMessage에 2개 넣었지만 parallel_tool_calls=False 설정이므로
    # 실제 모델은 보통 1개씩 호출합니다. Stub에서는 tools 노드가 pending 전체를 순회로 처리합니다.
    message("O004는 반품 가능합니다. 단순변심 시 반품비 3000원이 차감됩니다. 반품 신청을 진행할까요?"),
])

# parallel이 아닌 순차 호출로 다시
stub = Stub([
    routed("refund_exchange"),
    message("", ("get_order", {"order_id": "O004"})),
    message("", ("check_refund_eligibility", {"order_id": "O004"})),
    message("O004는 반품 가능합니다. 단순변심 시 반품비 3000원이 차감됩니다. 반품 신청을 진행할까요?"),
])
shop = Shop(BASE, "C01")
user0 = SCENARIOS["single-18"]["turns"][0]["user"]
result0 = run_turn(Agents(stub, "C01"), shop, 0, user0, [])

print("user:", user0)
print("routes:", result0["routes"], "reply:", result0["reply"])
print("tools:", [(t["name"], t["arguments"]) for t in shop.trace])
print("db status still:", shop.db["orders"]["O004"]["status"])  # 아직 신청 전

In [ ]:
# 예 C: 동의 후 반품 신청 (single-18 turn 1) — history에 이전 턴 메시지 유지
history = result0["messages"]
stub = Stub([
    routed("refund_exchange"),
    message("", ("request_refund", {"order_id": "O004", "reason": "단순변심"})),
    message("O004 반품을 접수했습니다. 환불 예정 금액은 52000원입니다."),
])
user1 = SCENARIOS["single-18"]["turns"][1]["user"]
result1 = run_turn(Agents(stub, "C01"), shop, 1, user1, history)

print("user:", user1)
print("reply:", result1["reply"])
print("order status:", shop.db["orders"]["O004"]["status"])
print("pending_refund:", shop.db["payments"]["O004"]["pending_refund"])

In [ ]:
# 예 D: handoff — 배송 담당에서 슈퍼바이저로 되돌린 뒤 반품 담당으로
stub = Stub([
    routed("order_delivery"),
    message("", ("transfer_back_to_supervisor", {})),
    routed("refund_exchange"),
    message("반품 담당입니다. 주문번호를 알려주세요."),
])
result = run_turn(Agents(stub, "C01"), Shop(BASE, "C01"), 0, "배송 문의인데 그냥 환불할래요", [])
print("routes:", result["routes"])  # ['order_delivery', 'refund_exchange']
print("transfers:", result["transfers"], "reply:", result["reply"])

## Step 5. grader로 채점

`grade_turn` / `grade_scenario` 는 route, 도구, 위반, 근거성, DB final_state를 봅니다.
시나리오 성공 조건(요약): final_state 일치 + 위반 0 + must_include 충족 + must_not_include 위반 0 + loop_limit 없음.

In [ ]:
# single-18 전체를 Stub으로 다시 돌리고 채점
# grade_scenario(scenario, rows, shop) — rows는 run_turn 결과에 calls/status를 붙인 것
scenario = SCENARIOS["single-18"]
shop = Shop(BASE, scenario["customer_id"], scenario.get("db_patch"), scenario.get("faults"))
history = []
rows = []

def pack(result, shop, offset):
    """run.py / tests 와 동일: 이 턴에서 새로 생긴 shop.trace 조각만 calls로 넣음"""
    return {**result, "calls": shop.trace[offset:], "status": "complete"}

# turn 0: 조회 + 동의 질문 (refund 실행 금지)
offset = len(shop.trace)
stub0 = Stub([
    routed("refund_exchange"),
    message("", ("get_order", {"order_id": "O004"})),
    message("", ("check_refund_eligibility", {"order_id": "O004"})),
    message("O004 반품 가능합니다. 단순변심이면 반품비 3000원이 차감됩니다. O004 단순변심 반품 신청을 진행할까요?"),
])
r0 = run_turn(Agents(stub0, scenario["customer_id"]), shop, 0, scenario["turns"][0]["user"], history)
rows.append(pack(r0, shop, offset))
history = r0["messages"]

# turn 1: 동의 후 신청 — reply_must_include: 반품, 검수
offset = len(shop.trace)
stub1 = Stub([
    routed("refund_exchange"),
    message("", ("request_refund", {"order_id": "O004", "reason": "단순변심"})),
    message("O004 반품을 접수했습니다. 환불 예정 금액은 52000원이며 반품 검수 대기입니다."),
])
r1 = run_turn(Agents(stub1, scenario["customer_id"]), shop, 1, scenario["turns"][1]["user"], history)
rows.append(pack(r1, shop, offset))

graded = grade_scenario(scenario, rows, shop)
print("success:", graded["success"], "complete:", graded["complete"])
print("final_state:", graded["final_state"])
print("violations:", graded["violations"])
for i, g in enumerate(graded["turns"]):
    print(f"turn {i}: route={g['route_correct']} recall={g['required_call_recall']} conf={g['confirmation_correct']} missing={g['include_missing']}")


In [ ]:
# 턴별 상세
for i, g in enumerate(graded["turns"]):
    print(f"=== turn {i} ===")
    print(json.dumps(g, ensure_ascii=False, indent=2)[:2500])


In [ ]:
# 일부러 틀린 Stub: 동의 전에 request_refund → unconfirmed_action 관측
scenario = SCENARIOS["single-18"]
shop_bad = Shop(BASE, "C01")
stub_bad = Stub([
    routed("refund_exchange"),
    message("", ("request_refund", {"order_id": "O004", "reason": "단순변심"})),  # 동의 전 실행
    message("반품 접수했습니다. 검수 대기입니다."),
    message("반품 접수했습니다. 검수 대기입니다."),  # 도구 강제 재시도용 응답 1개 추가
])
r_bad = run_turn(Agents(stub_bad, "C01"), shop_bad, 0, scenario["turns"][0]["user"], [])
row_bad = {**r_bad, "calls": shop_bad.trace[:], "status": "complete"}
# turn 1이 없어서 complete=False / success=False
g_bad = grade_scenario(scenario, [row_bad], shop_bad)
print("success:", g_bad["success"], "complete:", g_bad["complete"])
print("violations:", g_bad["violations"])
print("turn0 forbidden/unconfirmed detail:", g_bad["turns"][0]["violations"], "forbidden:", g_bad["turns"][0]["forbidden_calls"])
print("db after bad:", shop_bad.db["orders"]["O004"]["status"])


## Step 6. (선택) 실제 llama-server로 smoke 1개

**GPU가 비어 있고, 다른 서버가 없을 때만** 아래를 실행하세요.

- config: `g12-native` (Gemma 12B가 supervisor+답변 전부)
- 시나리오 1개만 (`single-01` 기본)
- 끝나면 서버를 반드시 내립니다 (`with resident(...)`)

원치 않으면 이 섹션은 건너뛰면 됩니다.

In [ ]:
RUN_REAL = False  # True로 바꿔야 실제 서버가 뜸

if not RUN_REAL:
    print("RUN_REAL=False — 실제 서버 셀 건너뜀. True로 바꾸면 g12-native로 1시나리오 실행.")
else:
    from shop.servers import resident
    from shop.agents import LocalLLM

    SID = "single-01"
    scenario = SCENARIOS[SID]
    port = 18380
    folder = ROOT / "runs" / "_walkthrough-scratch"
    folder.mkdir(parents=True, exist_ok=True)
    records = []

    with resident("g12-native", ngl=99, folder=folder, port=port, timeout=300, records=records) as (server, backend):
        llm = LocalLLM(server.port)
        try:
            shop = Shop(BASE, scenario["customer_id"], scenario.get("db_patch"), scenario.get("faults"))
            agents = Agents(llm, scenario["customer_id"], backend=backend)
            history = []
            rows = []
            for i, turn in enumerate(scenario["turns"]):
                print(f"turn {i}: {turn['user']}")
                offset = len(shop.trace)
                out = run_turn(agents, shop, i, turn["user"], history)
                history = out["messages"]
                rows.append({**out, "calls": shop.trace[offset:], "status": "complete"})
                print("  route:", out["routes"], "reply:", out["reply"][:200])
            graded = grade_scenario(scenario, rows, shop)
            print("success:", graded.get("success"))
            print("violations:", graded.get("violations"))
        finally:
            llm.close()
    print("server stopped. records:", records)


## 실험 아이디어

- Step 2: `cancel_order`를 배송중 주문에 호출해 `ALREADY_SHIPPED` 거부 확인, `cancel_items`로 일부 취소 해보기
- Step 4: Stub에서 `request_refund`를 turn 0에 넣어 동의 전 실행 재현
- Step 4: `human` 라우트로 `escalate` 노드 경로 보기
- Step 5: `reply_must_include`를 일부러 빠뜨린 답변으로 채점 실패 확인
- Step 6: `SID`를 `single-07` / `single-18`로 바꿔 실제 모델 행동 비교

관련 소스:
- `shop/tools.py` — Shop / schemas
- `shop/agents.py` — LocalLLM / Agents.supervise / respond
- `shop/graph.py` — StateGraph / run_turn
- `shop/grader.py` — grade_turn / grade_scenario
- `shop/servers.py` — resident / CONFIGS